In [1]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from pathlib import Path

In [2]:
import dagshub
dagshub.init(repo_owner='AMR-ITH', repo_name='RealEstateInsights', mlflow=True)
import mlflow

# set the tracking server

mlflow.set_tracking_uri("https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow")

# mlflow experiment

mlflow.set_experiment("Exp 3 - RF-HP Tuning-30")

Accessing as AMR-ITH

Initialized MLflow to track repo "AMR-ITH/RealEstateInsights"

Repository AMR-ITH/RealEstateInsights initialized!

<Experiment: artifact_location='mlflow-artifacts:/977cb1ccd3cd4a8198e4876b6a083c11', creation_time=1746527225080, experiment_id='5', last_update_time=1746527225080, lifecycle_stage='active', name='Exp 3 - RF-HP Tuning-30', tags={}>

In [4]:
# pathlib is a module in Python that provides an object-oriented interface 
# for working with file system paths.
current = Path.cwd()
parent = current.parent

# load the data 
df = pd.read_csv(parent /'data-scraped/interim_data.csv')
df.head()



,apartment_name,appartment_loc,zone,bhk_type,construction_status,carpet_area,bulit_area,super_bulit_area,price_value,nearbylocation,facility,luxury_facility_scores
0,nambiar millennia,Sarjapur Road,east,1,Under Construction,NaN,668.0,NaN,0.53,"[('mahatma vidhyalaya', '400 m'), ('eterssrt m...","['Yoga/Meditation Area', ""Children's Play Area...",69
1,provident capella,Samethanahalli,east,1,New Property,NaN,431.0,480.0,0.55,"[('soukya road', '1.4 km'), ('mvj college of e...","[""Children's Play Area"", 'Creche/Day Care', 'J...",70
2,brigade citre budigere cross,Byrathi,east,1,New Property,NaN,619.0,689.0,0.69,"[('one world international school', '3.2kms'),...","['Pet Park', ""Children's Play Area"", 'Landscap...",50
3,sattva east crest bandapura,Budigere Cross,east,1,New Property,NaN,537.0,598.0,0.70,"[('prerana international school', '700 m'), ('...","['Banquet Hall', 'Creche/Day Care', ""Children'...",74
4,sowparnika columns,Soukya Road,east,1,New Property,486.0,619.0,736.0,0.52,"['Whitefield Kadugodi Metro Station', 'Nexus S...","['Lift(s)', 'Swimming Pool', 'Park', 'Fitness ...",44


In [5]:
def categorize_luxury(score):
    if 0 <= score < 50:
        return 'low'
    elif 50 <= score < 150:
        return 'medium'
    else:
        return 'high'
    
df['luxury_category'] = df['luxury_facility_scores'].apply(categorize_luxury)

In [6]:
df.drop(columns=['carpet_area','super_bulit_area','nearbylocation','facility','apartment_name','appartment_loc','luxury_facility_scores'], inplace=True)

In [7]:
temp_df = df.copy()

X = temp_df.drop(columns=['price_value'])
y = temp_df['price_value']

In [8]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [9]:
# do the basic processing input data

num_cols = ['bulit_area']
nomial_cols = ['zone']
ordinal_cols = ['construction_status','bhk_type','luxury_category']


In [10]:
bhk_type_order = ['1','2','3','4','5','6','7','8','9','10']
construction_status_order = ['New Property','Under Construction', 'Relatively New', 'Moderatly Old', 'Old','undefined']
luxury_facility_scores_order = ['low','medium','high']

In [11]:
# build a preprocessor

prepocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
        ("nominal_encode", OneHotEncoder(handle_unknown="ignore",sparse_output=False), nomial_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[construction_status_order,bhk_type_order,luxury_facility_scores_order]), ordinal_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

prepocessor.set_output(transform="pandas")

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(), ['bulit_area']),
                                ('nominal_encode',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['zone']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['New Property',
                                                             'Under '
                                                             'Construction',
                                                             'Relatively New',
                                                             'Moderatly Old',
                                                             'Old',
                                                             'undefined'],
                                                            ['1', '2', '3', '4',
                                                             '5', '6', '7', '8',
                                                             '9', '10'],
                                                            ['low', 'medium',
                                                             'high']]),
                                 ['construction_status', 'bhk_type',
                                  'luxury_category'])],
                  verbose_feature_names_out=False)

In [12]:
# transform the data

X_train_trans = prepocessor.fit_transform(X_train)
X_test_trans = prepocessor.transform(X_test)

X_train_trans

,bulit_area,zone_east,zone_north,zone_south,zone_west,construction_status,bhk_type,luxury_category
2842,0.197885,0.0,1.0,0.0,0.0,0.0,2.0,1.0
903,0.185153,1.0,0.0,0.0,0.0,2.0,2.0,1.0
3262,0.297475,0.0,1.0,0.0,0.0,2.0,2.0,1.0
109,0.114804,1.0,0.0,0.0,0.0,1.0,1.0,1.0
5602,0.092361,0.0,0.0,0.0,1.0,5.0,1.0,0.0
...,...,...,...,...,...,...,...,...
3772,0.224860,0.0,1.0,0.0,0.0,0.0,3.0,1.0
5191,0.140160,0.0,0.0,1.0,0.0,0.0,2.0,1.0
5226,0.175874,0.0,0.0,1.0,0.0,4.0,2.0,1.0
5390,0.118041,0.0,0.0,0.0,1.0,2.0,1.0,1.0


In [13]:
from sklearn.ensemble import RandomForestRegressor
import optuna

from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score

c:\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [50]:
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
import time

In [24]:


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
        "random_state": 42,
        "n_jobs": -1,
    }
    
    # build the model
    lgbm = LGBMRegressor(**params)
    
    # train the model
    lgbm.fit(X_train_trans, y_train)
    
    # get the predictions
    y_pred_train = lgbm.predict(X_train_trans)
    y_pred_test = lgbm.predict(X_test_trans)

    
    # Calculate metrics
    mae_test = mean_absolute_error(y_test, y_pred_test)
    r2_test = r2_score(y_test, y_pred_test)

    # perform cross validation for MAE
    cv_mae_scores_train = cross_val_score(
        lgbm,
        X_train_trans,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    
    cv_mae_scores_test = cross_val_score(
        lgbm,
        X_test_trans,
        y_test,
        cv=5,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )
    
    # perform cross validation for R2
    cv_r2_scores_train = cross_val_score(
        lgbm,
        X_train_trans,
        y_train,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    cv_r2_scores_test = cross_val_score(
        lgbm,
        X_test_trans,
        y_test,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )
    
    
    # mean scores from cross-validation train
    mean_cv_mae_train = -cv_mae_scores_train.mean()
    mean_cv_r2_train = cv_r2_scores_train.mean()

    mean_cv_mae_test = -cv_mae_scores_test.mean()
    mean_cv_r2_test = cv_r2_scores_test.mean() 
    print("train mae",mean_cv_mae_train)
    print("train r2",mean_cv_r2_train)

    print("test mae",mean_cv_mae_test)
    print("test r2",mean_cv_r2_test)   
    
    # Store metrics in trial user attributes for display
    trial.set_user_attr("mae_test", mae_test)
    trial.set_user_attr("r2_test", r2_test)
    trial.set_user_attr("cv_mae_train", mean_cv_mae_train)
    trial.set_user_attr("cv_r2_train", mean_cv_r2_train)
    trial.set_user_attr("cv_mae_test",mean_cv_mae_test)
    trial.set_user_attr("cv_mae_test",mean_cv_r2_test)
    
    # Combined score - we want to minimize MAE and maximize R2
    # Lower value is better
    combined_score = mae_test / (1 + max(0, r2_test))
    
    return combined_score



In [27]:
# optimize the objective function
study.optimize(objective, n_trials=30, n_jobs=-1, show_progress_bar=True)






Best trial: 17. Best value: 0.250019:   3%|▎         | 1/30 [00:35<17:07, 35.41s/it]

train mae 0.5179493878254922
train r2 0.7919859618000679
test mae 0.5687820078189965
test r2 0.7438664323126739
[I 2025-05-08 17:50:05,940] Trial 43 finished with value: 0.27331670053544543 and parameters: {'n_estimators': 182, 'max_depth': 36, 'learning_rate': 0.08623651551553785, 'num_leaves': 58, 'min_child_samples': 92, 'subsample': 0.5023564016642542, 'colsample_bytree': 0.7816565741300607, 'reg_alpha': 7.6098046169437215, 'reg_lambda': 9.07841121226063}. Best is trial 17 with value: 0.2500185449653238.


Best trial: 17. Best value: 0.250019:   7%|▋         | 2/30 [00:36<07:00, 15.03s/it]

train mae 0.5129678565698365
train r2 0.7974721649444667
test mae 0.553117137699578
test r2 0.7620264981756553
[I 2025-05-08 17:50:06,715] Trial 48 finished with value: 0.26722258747500877 and parameters: {'n_estimators': 186, 'max_depth': 37, 'learning_rate': 0.15293223632802871, 'num_leaves': 111, 'min_child_samples': 49, 'subsample': 0.5645163953925569, 'colsample_bytree': 0.7865228480056493, 'reg_alpha': 9.250632056947799, 'reg_lambda': 4.172941880283828}. Best is trial 17 with value: 0.2500185449653238.


Best trial: 17. Best value: 0.250019:  10%|█         | 3/30 [00:37<04:00,  8.90s/it]

...

train mae 0.5119955714918782
train r2 0.7993199813282864
test mae 0.5511627540937183
test r2 0.7721677775158866
[I 2025-05-08 17:52:16,929] Trial 67 finished with value: 0.26547161555709603 and parameters: {'n_estimators': 223, 'max_depth': 14, 'learning_rate': 0.2558616645602224, 'num_leaves': 38, 'min_child_samples': 28, 'subsample': 0.7265719988169806, 'colsample_bytree': 0.660789513692106, 'reg_alpha': 7.595394877398239, 'reg_lambda': 7.238217330380452}. Best is trial 17 with value: 0.2500185449653238.


Best trial: 17. Best value: 0.250019:  93%|█████████▎| 28/30 [02:46<00:02,  1.01s/it]

train mae 0.5129033894240589
train r2 0.7996123303271319
test mae 0.5517374797729705
test r2 0.7698413911712352
[I 2025-05-08 17:52:17,340] Trial 68 finished with value: 0.26785785107737636 and parameters: {'n_estimators': 237, 'max_depth': 14, 'learning_rate': 0.23079551976639556, 'num_leaves': 21, 'min_child_samples': 27, 'subsample': 0.7693401402078017, 'colsample_bytree': 0.6586737811824726, 'reg_alpha': 7.6963017286613695, 'reg_lambda': 6.806496310417832}. Best is trial 17 with value: 0.2500185449653238.
train mae 0.5072023165036408
train r2 0.7991239366872576
test mae 0.5487592362475564
test r2 0.7687400537561336
[I 2025-05-08 17:52:17,508] Trial 69 finished with value: 0.26754319197246074 and parameters: {'n_estimators': 152, 'max_depth': 14, 'learning_rate': 0.23695331193503777, 'num_leaves': 125, 'min_child_samples': 30, 'subsample': 0.7396294112062634, 'colsample_bytree': 0.9898738663577338, 'reg_alpha': 8.380933379749257, 'reg_lambda': 2.6596428774273475}. Best is trial 17 w

Best trial: 17. Best value: 0.250019: 100%|██████████| 30/30 [02:47<00:00,  5.58s/it]

train mae 0.5186736349126139
train r2 0.7908219947254413
test mae 0.5745016021240866
test r2 0.7384776716590205
[I 2025-05-08 17:52:17,610] Trial 70 finished with value: 0.27305202538528855 and parameters: {'n_estimators': 144, 'max_depth': 14, 'learning_rate': 0.24966539755716194, 'num_leaves': 42, 'min_child_samples': 100, 'subsample': 0.7358594973106954, 'colsample_bytree': 0.6487520187942994, 'reg_alpha': 8.39196852645544, 'reg_lambda': 2.608606897298494}. Best is trial 17 with value: 0.2500185449653238.
train mae 0.5209396945602824
train r2 0.7913015933121736
test mae 0.5775696247957247
test r2 0.7375239281893241
[I 2025-05-08 17:52:17,782] Trial 71 finished with value: 0.27555232657176415 and parameters: {'n_estimators': 149, 'max_depth': 21, 'learning_rate': 0.24081678333638917, 'num_leaves': 44, 'min_child_samples': 100, 'subsample': 0.7439673762853298, 'colsample_bytree': 0.6871293612447112, 'reg_alpha': 9.89176992007244, 'reg_lambda': 3.1397824907765957}. Best is trial 17 wit

In [28]:
# train the model on best parameters
best_rf = LGBMRegressor(**study.best_params)

best_rf.fit(X_train_trans,y_train)

y_pred_train = best_rf.predict(X_train_trans)
y_pred_test = best_rf.predict(X_test_trans)



scores = cross_val_score(best_rf,X_train_trans,y_train,cv=5,n_jobs=-1)

# mae,r2 for test and train
mae_train = mean_absolute_error(y_train,y_pred_train)
r2_train = r2_score(y_train,y_pred_train)
mae_test = mean_absolute_error(y_test,y_pred_test)
r2_test = r2_score(y_test,y_pred_test)


print(mae_train)
print(r2_train)
print(mae_test)
print(r2_test)

[LightGBM] [Warning] Unknown parameter: max_samples
[LightGBM] [Warning] Unknown parameter: max_features
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Warning] min_data_in_leaf is set with min_child_samples=20, will be overridden by min_samples_leaf=1. Current value: min_data_in_leaf=1
[LightGBM] [Warning] Unknown parameter: bootstrap
[LightGBM] [Warning] Unknown parameter: max_samples
[LightGBM] [Warning] Unknown parameter: max_features
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Warning] min_data_in_leaf is set with min_child_samples=20, will be overridden by min_samples_leaf=1. Current value: min_data_in_leaf=1
[LightGBM] [Warning] Unknown parameter: bootstrap
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000371 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 470
[LightGBM] [Info] Number of data points in the train set: 4860, number of used feat

In [18]:
study.best_params

{'n_estimators': 150,
 'max_depth': 25,
 'learning_rate': 0.22571315117777171,
 'num_leaves': 94,
 'min_child_samples': 48,
 'subsample': 0.7859608525116539,
 'colsample_bytree': 0.6713298072955733,
 'reg_alpha': 2.9441365937007995,
 'reg_lambda': 8.073108740850945}

In [19]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 10, 500),
            "max_depth": trial.suggest_int("max_depth", 1, 30),
            "max_features": trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_samples": trial.suggest_float("max_samples", 0.5, 1),
            "bootstrap": trial.suggest_categorical("bootstrap", [True]),  # Always True if using max_samples
            "random_state": 42,
            "n_jobs": -1,
        }

        # Log hyperparameters
        mlflow.log_params(params)

        # Build and train the model
        rf = RandomForestRegressor(**params)
        rf.fit(X_train_trans, y_train)

        # Predictions
        y_pred_train = rf.predict(X_train_trans)
        y_pred_test = rf.predict(X_test_trans)

        # Basic evaluation
        mae_test = mean_absolute_error(y_test, y_pred_test)
        r2_test = r2_score(y_test, y_pred_test)

        # Cross-validation on train data
        cv_mae_scores_train = cross_val_score(
            rf, X_train_trans, y_train, cv=5,
            scoring="neg_mean_absolute_error", n_jobs=-1
        )
        cv_r2_scores_train = cross_val_score(
            rf, X_train_trans, y_train, cv=5,
            scoring="r2", n_jobs=-1
        )

        # Cross-validation on test data (optional, but included for insight)
        cv_mae_scores_test = cross_val_score(
            rf, X_test_trans, y_test, cv=5,
            scoring="neg_mean_absolute_error", n_jobs=-1
        )
        cv_r2_scores_test = cross_val_score(
            rf, X_test_trans, y_test, cv=5,
            scoring="r2", n_jobs=-1
        )

        # Compute means
        mean_cv_mae_train = -cv_mae_scores_train.mean()
        mean_cv_r2_train = cv_r2_scores_train.mean()
        mean_cv_mae_test = -cv_mae_scores_test.mean()
        mean_cv_r2_test = cv_r2_scores_test.mean()

        # Log metrics
        mlflow.log_metric("cv_mae_train", mean_cv_mae_train)
        mlflow.log_metric("cv_r2_train", mean_cv_r2_train)
        mlflow.log_metric("cv_mae_test", mean_cv_mae_test)
        mlflow.log_metric("cv_r2_test", mean_cv_r2_test)
        mlflow.log_metric("test_mae", mae_test)
        mlflow.log_metric("test_r2", r2_test)

        # Print for tracking
        print("train mae:", mean_cv_mae_train)
        print("train r2:", mean_cv_r2_train)
        print("test mae:", mean_cv_mae_test)
        print("test r2:", mean_cv_r2_test)

        # Store Optuna user attributes
        trial.set_user_attr("mae_test", mae_test)
        trial.set_user_attr("r2_test", r2_test)
        trial.set_user_attr("cv_mae_train", mean_cv_mae_train)
        trial.set_user_attr("cv_r2_train", mean_cv_r2_train)
        trial.set_user_attr("cv_mae_test", mean_cv_mae_test)
        trial.set_user_attr("cv_r2_test", mean_cv_r2_test)

        # Final score to minimize
        # combined_score = mae_test / (1 + max(0, r2_test))
        # mlflow.log_metric("combined_score", combined_score)

        return mean_cv_mae_test


In [20]:
import optuna

In [21]:
# import mlflow

study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective,n_trials=30,n_jobs=-1,show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric("best_score",study.best_value)

    # train the model on best parameters
    best_rf = RandomForestRegressor(**study.best_params)

    best_rf.fit(X_train_trans,y_train)

    y_pred_train = best_rf.predict(X_train_trans)
    y_pred_test = best_rf.predict(X_test_trans)
    mlflow.sklearn.log_model(best_rf,"model")
    mlflow.log_params(best_rf.get_params())


    scores = cross_val_score(best_rf,X_train_trans,y_train,cv=5,n_jobs=-1)

    # mae,r2 for test and train
    mae_train = mean_absolute_error(y_train,y_pred_train)
    r2_train = r2_score(y_train,y_pred_train)
    mae_test = mean_absolute_error(y_test,y_pred_test)
    r2_test = r2_score(y_test,y_pred_test)

    mlflow.log_metric("mae-train",mae_train)
    mlflow.log_metric("r2-train",r2_train)
    mlflow.log_metric("mae-test",mae_test)
    mlflow.log_metric("r2-test",r2_test)

        # log the best model
    mlflow.sklearn.log_model(best_rf,artifact_path="model")

[I 2025-05-12 09:23:15,378] A new study created in memory with name: no-name-3c036d98-86d0-4ab0-a35f-cb2eda34a863
  0%|          | 0/30 [00:00<?, ?it/s]

train mae: 0.5270438662039435
train r2: 0.7864051502243737
test mae: 0.5752248226028305
test r2: 0.7424697166959036
train mae: 0.5058707208825063
train r2: 0.7977605567912323
test mae: 0.5301734856591336
test r2: 0.7802234666893904
🏃 View run vaunted-hog-614 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5/runs/bca44ea928b14fc88e2fab9524f9ff10
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5
🏃 View run overjoyed-skink-385 at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5/runs/8cacd35108cd4024a19db6e623c922f7
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5


Best trial: 1. Best value: 0.575225:   3%|▎         | 1/30 [00:29<14:02, 29.06s/it]

[I 2025-05-12 09:23:44,794] Trial 1 finished with value: 0.5752248226028305 and parameters: {'n_estimators': 245, 'max_depth': 12, 'max_features': 'sqrt', 'min_samples_split': 7, 'min_samples_leaf': 9, 'max_samples': 0.8997223324574495, 'bootstrap': True}. Best is trial 1 with value: 0.5752248226028305.


Best trial: 0. Best value: 0.530173:   7%|▋         | 2/30 [00:29<05:50, 12.52s/it]

...

[I 2025-05-12 09:28:04,738] Trial 28 finished with value: 0.5386752288888492 and parameters: {'n_estimators': 187, 'max_depth': 25, 'max_features': 'log2', 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_samples': 0.7185920447784611, 'bootstrap': True}. Best is trial 18 with value: 0.5248203902279704.


Best trial: 18. Best value: 0.52482: 100%|██████████| 30/30 [04:50<00:00,  9.67s/it]


[I 2025-05-12 09:28:05,736] Trial 29 finished with value: 0.5318413797569803 and parameters: {'n_estimators': 174, 'max_depth': 24, 'max_features': 'log2', 'min_samples_split': 3, 'min_samples_leaf': 2, 'max_samples': 0.7267083449920694, 'bootstrap': True}. Best is trial 18 with value: 0.5248203902279704.


2025/05/12 09:28:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/05/12 09:28:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run best_model at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5/runs/409dd7775c9045fda7daa0b558514b55
🧪 View experiment at: https://dagshub.com/AMR-ITH/RealEstateInsights.mlflow/#/experiments/5


In [22]:
study.best_params

{'n_estimators': 150,
 'max_depth': 29,
 'max_features': 'log2',
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_samples': 0.5052045211887413,
 'bootstrap': True}

In [23]:
# train the model on best parameters
best_rf = RandomForestRegressor(**study.best_params)

best_rf.fit(X_train_trans,y_train)

y_pred_train = best_rf.predict(X_train_trans)
y_pred_test = best_rf.predict(X_test_trans)



scores = cross_val_score(best_rf,X_train_trans,y_train,cv=5,n_jobs=-1)

# mae,r2 for test and train
mae_train = mean_absolute_error(y_train,y_pred_train)
r2_train = r2_score(y_train,y_pred_train)
mae_test = mean_absolute_error(y_test,y_pred_test)
r2_test = r2_score(y_test,y_pred_test)


print(mae_train)
print(r2_train)
print(mae_test)
print(r2_test)

0.3076572094211611
0.9242606163424838
0.4657499930855609
0.8333173218097839
